# Large-Array Cascade Demo

Demonstrates partitioning large cylinder arrays into groups, computing each group's
S-matrix independently, then cascading them together.

**Why cascade?** Computing the S-matrix scales as $O(N^3)$ with the number of cylinders.
By splitting $N$ cylinders into $G$ groups of $N/G$, the cost drops from $O(N^3)$ to
$G \cdot O((N/G)^3)$ — much faster for large $N$.

In [ ]:
import sys
import time
import numpy as np
from numpy.linalg import svd

sys.path.insert(0, '../..')

from Scattering_Code.smatrix_parameters import smatrix_parameters
from Scattering_Code.smatrix import smatrix
from Scattering_Code.cascadertwo import cascadertwo

In [ ]:
WAVELENGTH = 0.93
PERIOD     = 12.81
RADIUS     = 0.25
MU         = 1.0
CMMAX      = 5
PHIINC     = np.pi / 2
EPSILON    = 1.3**2
SEED       = 42

nmax = int(np.floor(PERIOD / WAVELENGTH))
nm   = 2 * nmax + 1

## 1. Generate Large Cylinder Array

In [ ]:
NUM_CYL    = 100
NUM_GROUPS = 5

thickness = NUM_CYL * 0.3

rng = np.random.RandomState(SEED)
margin = RADIUS * 2.0
min_sep = 2.5 * RADIUS
clocs = np.zeros((NUM_CYL, 2))

for i in range(NUM_CYL):
    for _ in range(10000):
        x = margin + rng.rand() * (PERIOD - 2*margin)
        y = margin + rng.rand() * (thickness - 2*margin)
        if i == 0 or np.all(np.sqrt((x - clocs[:i, 0])**2 + (y - clocs[:i, 1])**2) > min_sep):
            clocs[i] = [x, y]
            break

print(f"{NUM_CYL} cylinders placed in slab of thickness {thickness}")

## 2. Divide Into Groups by Y-Position

In [ ]:
# Sort by y-position and split into groups
sorted_idx = np.argsort(clocs[:, 1])
sorted_clocs = clocs[sorted_idx]

group_size = int(np.ceil(NUM_CYL / NUM_GROUPS))
groups = []
for g in range(NUM_GROUPS):
    start = g * group_size
    end = min((g+1) * group_size, NUM_CYL)
    if start < NUM_CYL:
        groups.append(sorted_clocs[start:end])

for i, g in enumerate(groups):
    print(f"  Group {i+1}: {len(g)} cylinders, y range [{g[:,1].min():.1f}, {g[:,1].max():.1f}]")

## 3. Compute S-Matrix for Each Group and Cascade

In [ ]:
S_matrices = []
d_values = []
total_time = 0

for i, group_clocs in enumerate(groups):
    n_cyl = len(group_clocs)
    
    # Shift y to start from 0 + margin
    y_min = group_clocs[:, 1].min()
    shifted = group_clocs.copy()
    shifted[:, 1] -= (y_min - RADIUS * 2)
    
    d = group_clocs[:, 1].max() - group_clocs[:, 1].min() + 4 * RADIUS
    
    sp = smatrix_parameters(WAVELENGTH, PERIOD, PHIINC,
                            1e-11, 1e-4, 5, 3, 1000, 3, 5, 1, PERIOD/120)
    cmmaxs_g = np.full(n_cyl, CMMAX, dtype=int)
    cepmus_g = np.column_stack([np.full(n_cyl, EPSILON), np.full(n_cyl, MU)])
    crads_g  = np.full(n_cyl, RADIUS)
    
    print(f"Group {i+1}/{len(groups)}: {n_cyl} cylinders, d={d:.2f}...", end=" ")
    t0 = time.time()
    S_g, _ = smatrix(shifted, cmmaxs_g, cepmus_g, crads_g, PERIOD, WAVELENGTH,
                      nmax, d, sp, 'Off')
    dt = time.time() - t0
    total_time += dt
    print(f"{dt:.1f}s")
    
    S_matrices.append(S_g)
    d_values.append(d)

# Cascade
print("\nCascading...")
t0 = time.time()
S_cas = S_matrices[0]
d_cas = d_values[0]
for i in range(1, len(S_matrices)):
    S_cas, d_cas = cascadertwo(S_cas, d_cas, S_matrices[i], d_values[i])
    print(f"  Cascaded 1-{i+1}, total d={d_cas:.1f}")

cascade_time = time.time() - t0
print(f"\nGroup computation: {total_time:.1f}s")
print(f"Cascade: {cascade_time:.3f}s")
print(f"Total: {total_time + cascade_time:.1f}s")

## 4. Transmission Results

In [ ]:
S11 = S_cas[:nm, :nm]
S21 = S_cas[nm:, :nm]

center = nmax
R = np.sum(np.abs(S11[:, center])**2)
T = np.sum(np.abs(S21[:, center])**2)

print(f"S-matrix size: {S_cas.shape}")
print(f"Total thickness: {d_cas:.1f}")
print(f"Reflection (R): {R:.6f}")
print(f"Transmission (T): {T:.6f}")
print(f"R + T: {R+T:.6f}")